# Automatic Differentiation with RustQuant (Python)

RustQuant's Rust core provides reverse-mode automatic differentiation (AAD).
From Python, we access derivatives via the `BlackScholesMerton` Greeks API,
and demonstrate the underlying concepts with finite differences.

See the **Rust kernel** version (`01_automatic_differentiation.ipynb`) for
direct use of the `Graph`, `Variable`, and `accumulate()` API.

## Setup

In [ ]:
import math
from RustQuant.instruments import BlackScholesMerton, OptionType

## 1. Finite Differences vs Analytic Derivatives

Automatic differentiation computes exact derivatives in one pass.
Here we compare analytic Greeks (from RustQuant) with numerical finite differences.

In [ ]:
def make_bsm(S=100.0, K=100.0, vol=0.20, r=0.05, opt=OptionType.Call):
    return BlackScholesMerton(
        underlying_price=S, strike_price=K, volatility=vol,
        risk_free_rate=r, cost_of_carry=r,
        expiry_year=2027, expiry_month=3, expiry_day=22,
        option_type=opt,
    )

bsm = make_bsm()

print(f"Call Price = {bsm.price():.6f}")
print(f"Delta      = {bsm.delta():.6f}")
print(f"Gamma      = {bsm.gamma():.6f}")
print(f"Vega       = {bsm.vega():.6f}")
print(f"Theta      = {bsm.theta():.6f}")
print(f"Rho        = {bsm.rho():.6f}")

## 2. Numerical Differentiation (Central Differences)

For any function $f(x)$, the central difference approximation is:

$$f'(x) \approx \frac{f(x+h) - f(x-h)}{2h}$$

This is $O(h^2)$ accurate but requires **2 function evaluations per parameter**.
Autodiff computes all derivatives in a single forward+reverse pass.

In [ ]:
h = 1e-4

# Delta: dC/dS
delta_fd = (make_bsm(S=100+h).price() - make_bsm(S=100-h).price()) / (2*h)

# Gamma: d²C/dS²
gamma_fd = (make_bsm(S=100+h).price() - 2*bsm.price() + make_bsm(S=100-h).price()) / (h**2)

# Vega: dC/dvol
vega_fd = (make_bsm(vol=0.20+h).price() - make_bsm(vol=0.20-h).price()) / (2*h)

# Rho: dC/dr
rho_fd = (make_bsm(r=0.05+h).price() - make_bsm(r=0.05-h).price()) / (2*h)

print(f"{'Greek':<8} {'Analytic':>12} {'Finite Diff':>12} {'Abs Error':>12}")
print("-" * 44)
print(f"{'Delta':<8} {bsm.delta():>12.6f} {delta_fd:>12.6f} {abs(bsm.delta()-delta_fd):>12.2e}")
print(f"{'Gamma':<8} {bsm.gamma():>12.6f} {gamma_fd:>12.6f} {abs(bsm.gamma()-gamma_fd):>12.2e}")
print(f"{'Vega':<8} {bsm.vega():>12.6f} {vega_fd:>12.6f} {abs(bsm.vega()-vega_fd):>12.2e}")
print(f"{'Rho':<8} {bsm.rho():>12.6f} {rho_fd:>12.6f} {abs(bsm.rho()-rho_fd):>12.2e}")

## 3. All Greeks at Once

RustQuant computes all 13+ Greeks in one call, equivalent to a single
autodiff reverse pass — much faster than bumping each parameter.

In [ ]:
greeks = bsm.greeks()

for name, value in greeks.items():
    print(f"  {name:<10} = {value:>12.6f}")

## 4. Delta Surface Across Spot and Strike

Visualising how Delta changes with moneyness.

In [ ]:
print(f"{'Spot':>6} {'K=90':>10} {'K=100':>10} {'K=110':>10}")
print("-" * 36)
for s in range(80, 125, 5):
    d90  = make_bsm(S=float(s), K=90.0).delta()
    d100 = make_bsm(S=float(s), K=100.0).delta()
    d110 = make_bsm(S=float(s), K=110.0).delta()
    print(f"{s:>6} {d90:>10.4f} {d100:>10.4f} {d110:>10.4f}")

## 5. Implied Volatility via Autodiff

Implied volatility solvers internally use derivatives (Newton-Raphson on vega).
RustQuant uses Jaeckel's "Let's Be Rational" method.

In [ ]:
market_price = bsm.price()
iv = bsm.implied_volatility(market_price)
print(f"Price from vol=0.20: {market_price:.6f}")
print(f"Recovered IV:        {iv:.6f}")

## Summary

| Method | # Evaluations for N params | Accuracy |
|--------|---------------------------|----------|
| Finite Differences | 2N | $O(h^2)$ |
| Forward-mode AD | N | Machine precision |
| **Reverse-mode AD** | **1** | **Machine precision** |

RustQuant uses reverse-mode AD internally. From Python, Greeks are accessed
via `BlackScholesMerton.delta()`, `.gamma()`, etc., or all at once via `.greeks()`.